# Step 0.2 - MiniMol chemical-structure embeddings

**What this notebook does.** It turns pre-computed MiniMol embeddings into a clean table - one row per
compound, one column per embedding dimension - annotated with JUMP compound metadata.

**Background.** [MiniMol](https://arxiv.org/abs/2404.14986) is a pretrained molecular "foundation
model": a graph neural network that reads a molecule's structure and outputs a vector of 512 numbers
(an *embedding*). Molecules with similar structures and properties receive similar vectors.
In this project MiniMol represents **chemical structure alone**, the baseline against which the
biological representations are compared.

**Input.** `minimol_embeddings__Kim_et_al_subset.csv`: MiniMol embeddings for the subset of JUMP
compounds used by Kim *et al.* (Bayer; see Step 0.3). Each row stores its embedding as a text list
such as `"[0.12, -0.53, …]"`.

**Output.** `minimol_embeddings__Kim_et_al_subset_final.parquet`, indexed by InChIKey.

**Where to run it.** Written for Google Colab / Colab Enterprise: it uses the `gsutil` command-line tool and
caches immutable downloads under the repository-level `data/raw/` directory and retains assembled local
outputs under this stage's `results/` directory. You need read/write access to the
`gs://calico-reference-embeddings` Cloud Storage bucket for the remote sources and uploads.

## Setup

All libraries used in this notebook are imported in the next cell. Locally they are installed by `uv sync`
(see the main README); on Google Colab, first run
`!pip install git+https://github.com/axiomcura/ModelBenchmarking.git`, which installs the same dependencies.

In [1]:
import ast

import numpy as np
import pandas as pd
from IPython.display import display

from model_benchmarking.paths import cache_gcs_file, notebook_artifact_dirs

RAW_DATA_DIR, RESULTS_DIR = notebook_artifact_dirs("0.download_and_assemble", "minimol")
METADATA_RAW_DIR, _ = notebook_artifact_dirs("0.download_and_assemble", "compound_metadata")


## Unpack the embeddings and attach compound metadata

### 1. Configuration
Input and output locations, the column holding the packed embeddings and the column used to match embeddings to metadata (`SMILES_standard`, a canonical text description of the structure).

In [2]:
# Inputs: immutable source downloads are cached under data/raw/.
minimol_gcs = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/minimol_embeddings/minimol_embeddings__Kim_et_al_subset.csv'
metadata_gcs = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'
minimol_csv = cache_gcs_file(minimol_gcs, RAW_DATA_DIR / 'minimol_embeddings__Kim_et_al_subset.csv')
metadata_csv = cache_gcs_file(metadata_gcs, METADATA_RAW_DIR / 'JUMP_CP_compound_metadata_with_SMILES_standard.csv')

# Outputs: retain a local result and upload the same table to GCS.
output_parquet = RESULTS_DIR / 'minimol_embeddings__Kim_et_al_subset_final.parquet'
gcs_output_parquet = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/minimol_embeddings/minimol_embeddings__Kim_et_al_subset_final.parquet'

EMB_COL = 'minimol'           # The column containing the packed embeddings
JOIN_COL = 'SMILES_standard'  # The column used to link embeddings and metadata

### 2. Load the embeddings and the metadata

In [3]:
print(f"📂 Loading Minimol embeddings...")
df_emb = pd.read_csv(minimol_csv)
print(f"   ✅ Loaded {len(df_emb):,} rows.")

print(f"📂 Loading Metadata...")
df_meta = pd.read_csv(metadata_csv)
print(f"   ✅ Loaded {len(df_meta):,} rows.")

### 3. Unpack the embeddings
Each text list is parsed and split into numeric columns `Mini_Dim_1 … Mini_Dim_512`. Empty or malformed rows are dropped.

In [4]:
if EMB_COL not in df_emb.columns:
    raise KeyError(f"❌ Could not find '{EMB_COL}'. Available columns: {list(df_emb.columns)}")

print(f"\n💥 Expanding the '{EMB_COL}' column...")

# Clean out any blank or broken embeddings to prevent crashes
df_emb = df_emb.dropna(subset=[EMB_COL])
if isinstance(df_emb[EMB_COL].iloc[0], str):
    df_emb = df_emb[df_emb[EMB_COL].str.strip() != ""]
    df_emb = df_emb[df_emb[EMB_COL].str.strip() != "[]"]
    print("   -> Parsing string representations into Python lists...")
    df_emb[EMB_COL] = df_emb[EMB_COL].apply(ast.literal_eval)

df_emb = df_emb[df_emb[EMB_COL].map(len) > 0]

# Fast extraction using numpy vectorization
print("   -> Splitting into separate float columns...")
expanded_matrix = np.vstack(df_emb[EMB_COL].values)
num_dimensions = expanded_matrix.shape[1]

dim_cols = [f"Mini_Dim_{i+1}" for i in range(num_dimensions)]
df_expanded = pd.DataFrame(expanded_matrix, columns=dim_cols, index=df_emb.index)

# Drop the old packed column and attach the expanded columns
df_emb = pd.concat([df_emb.drop(columns=[EMB_COL]), df_expanded], axis=1)
print(f"   ✅ Successfully expanded into {num_dimensions} distinct float columns!")

### 4. Attach metadata
An inner join on `SMILES_standard` keeps only molecules present in both tables.

In [5]:
print("\n⚙️ Joining metadata onto the expanded embeddings...")

if JOIN_COL not in df_emb.columns or JOIN_COL not in df_meta.columns:
    raise KeyError(f"❌ Join column '{JOIN_COL}' must exist in BOTH files!")

# Deduplicate metadata on the join column to avoid accidental row duplication
df_meta = df_meta.drop_duplicates(subset=[JOIN_COL])

# Inner join so we only keep molecules that have both embeddings and metadata
df_joined = pd.merge(df_emb, df_meta, on=JOIN_COL, how='inner')
print(f"   ✅ Join complete! Shape: {df_joined.shape[0]:,} Rows, {df_joined.shape[1]:,} Columns")

# Auto-detect the exact spelling of InChIKey (InChIKey vs Metadata_InChIKey)
INDEX_COL = next((col for col in df_joined.columns if 'inchikey' in col.lower()), None)
if not INDEX_COL:
    raise KeyError("❌ Could not find an InChIKey column in the joined data!")

print(f"🔑 Target index column identified as: '{INDEX_COL}'")

### 5. Index by InChIKey
The InChIKey - a fixed-length code derived from the chemical structure - is the compound identifier used throughout the project. Duplicates keep their first occurrence; text metadata columns come first, followed by the numeric embedding columns.

> **Note.** Columns are split into metadata and embeddings purely by data type, so any *numeric* metadata column would be placed among the embedding columns. Code that later selects "all numeric columns" as features should find exactly 512.

In [6]:
print(f"\n🧹 Cleaning data types and setting '{INDEX_COL}' as the primary index...")

# Drop rows where InChIKey is missing
df_joined = df_joined.dropna(subset=[INDEX_COL])
df_joined[INDEX_COL] = df_joined[INDEX_COL].astype(str).str.strip()

# Deduplicate by InChIKey (keeping the first instance)
df_joined = df_joined[~df_joined[INDEX_COL].duplicated(keep='first')]

# Set InChIKey as the primary index
df_joined.set_index(INDEX_COL, inplace=True)

# Rearrange columns to put the metadata text on the left, followed by the float dimensions
metadata_cols = df_joined.select_dtypes(exclude=['number']).columns.tolist()
embedding_cols = [c for c in df_joined.columns if c not in metadata_cols]
df_joined = df_joined[metadata_cols + embedding_cols]

# Prevent PyArrow Parquet crashes by forcing all metadata columns to be explicitly typed as strings
for col in metadata_cols:
    df_joined[col] = df_joined[col].astype(str)

### 6. Save

In [7]:
print("\n💾 Saving compressed Parquet locally and to Google Cloud Storage...")
df_joined.to_parquet(output_parquet, compression='snappy')
df_joined.to_parquet(gcs_output_parquet, compression='snappy')

print("🎉 SUCCESS! Files saved to:")
print(f"   Local: {output_parquet}")
print(f"   GCS:   {gcs_output_parquet}")

# Display the first 3 rows to verify the index, metadata, and expanded columns
print("\n--- PREVIEW OF ASSEMBLED PARQUET ---")
display(df_joined.head(3))